# 02 — Exploring a real corpus with the Explorer

This notebook points groundline at a production-scale corpus — the **458
with-sema parse-tree dumps of MOM6 + FMS2** — and browses it interactively with
the `Explorer` widget, then shows how to get the same neighbourhood data
programmatically (no widget, no browser) via `groundline.graph_view`.

Prerequisites: notebook **01** for the concepts (IR, scope-qualified identity,
confidence strata), and access to a corpus of dumps — by default the one on
NCAR's glade filesystem, overridable with the `GROUNDLINE_DUMPS` environment
variable.

> **Which corpus?** MOM6-with-FMS2 is currently the *only* real corpus. There is
> no TIM corpus yet: `bin/flang_ptree/MOM6_using_TIM/` is an empty skeleton,
> because the dump-only toolchain cannot build AMReX (see `docs/DEVLOG.md`,
> 2026-05-28). When a TIM corpus exists, this notebook should run over it
> unchanged via `GROUNDLINE_DUMPS`.

> **Kernel note:** launch Jupyter with `PYTHONNOUSERSITE=1` and the repo venv —
> see `notebooks/README.md`. If imports fail on a pandas/numpy error, a broken
> `~/.local` install is shadowing the venv.

## Parameters

`GROUNDLINE_DUMPS` may point at any directory tree containing
`-fdebug-dump-parse-tree` output files named `*_ptree` (searched recursively).

In [ ]:
import os
from pathlib import Path

CORPUS = Path(os.environ.get(
    "GROUNDLINE_DUMPS",
    "/glade/work/altuntas/turbo-stack/bin/flang_ptree/MOM6_using_FMS2",
))
# The entity whose neighbourhood the programmatic sections inspect.
FOCUS_ID = os.environ.get("GROUNDLINE_FOCUS", "fms_diag_bbox_mod::reset_bounds")

assert CORPUS.is_dir(), (
    f"{CORPUS} not found — set GROUNDLINE_DUMPS to a directory of *_ptree dumps."
)
paths = sorted(p for p in CORPUS.rglob("*_ptree") if p.is_file())
print(f"{len(paths)} parse-tree dumps under {CORPUS}")

## Build the forest (~40 s for the full corpus)

A handful of "does not start with proper header" warnings are expected: the FMS
build compiles some C sources, and their `*.o_ptree` files are not Fortran parse
trees — the frontend skips them and moves on (fault isolation, one bad file
never aborts the forest).

In [ ]:
from groundline.parse_forest import ParseForest

forest = ParseForest(paths)
ir = forest.ir

## Fact-base summary

The call relation is stored **stratified by confidence** (D3): `calls_resolved`
/ `calls_assumed` / `calls_unresolved` are the pure relations; `calls` (may) and
`calls_must` (must) are computed views over them. Unresolved *targets* are
first-class entities with `defined=False` — the count below is how many names
this corpus references but never defines (external libraries such as netcdf and
mpi, plus intrinsics the frontend's list misses).

In [ ]:
undefined = sum(1 for e in ir.entities.values() if not e.defined)
print(f"entities:    {len(ir.entities)}  ({undefined} referenced-but-undefined)")
print(f"modules:     {len(ir.modules)}   subroutines: {len(ir.subroutines)}   "
      f"functions: {len(ir.functions)}")
print(f"interfaces:  {len(ir.interfaces)}   derived types: {len(ir.derived_types)}")
print()
print(f"calls_resolved:   {len(ir.calls_resolved):6}")
print(f"calls_assumed:    {len(ir.calls_assumed):6}")
print(f"calls_unresolved: {len(ir.calls_unresolved):6}")
print(f"may  (calls):      {len(ir.calls):6}")
print(f"must (calls_must): {len(ir.calls_must):6}")
print(f"file errors:      {len(ir.file_errors):6}")

## The interactive Explorer

Pick a category, search, select an entity — the widget draws its one-hop
neighbourhood. The visual encoding (spelled out in the legend):

* edge **line style** = confidence stratum: solid `resolved`, dashed `assumed`,
  dotted + muted `unresolved`;
* edge **colour** = direction relative to the selected node (blue in, red out);
* purple diamond-headed edges = generic-interface membership (structure, not a
  call, so no confidence);
* **ghosted nodes** = `defined=False` targets — referenced but never parsed;
* nodes are grouped into their enclosing modules.

Selector entries are scope-qualified ids, so same-named routines in different
modules (e.g. several `initialize`s) stay distinct entries.

In [ ]:
from groundline.explorer import Explorer

Explorer(forest)

## Programmatic lookups, the seam way

Everything the widget shows is answerable directly from the IR — by
scope-qualified id, never by bare name.

In [ ]:
ent = ir.get(FOCUS_ID)
print(ent.kind, ent.id, "| scope:", ent.scope, "| defined:", ent.defined)
print("callers:", sorted(c.id for c in ir.callers(ent.id))[:10])
print("callees:", sorted(c.id for c in ir.callees(ent.id))[:10])

## Neighbourhoods without the widget: `groundline.graph_view`

`graph_view` is the pure half of the Explorer: IR + a center entity → a
NetworkX neighbourhood (`gen_subgraph`) → cytoscape-shaped element dicts
(`subgraph_elements`). No ipywidgets, no browser — the same content decisions
the widget renders, usable from scripts, tests, or headless pipelines.

In [ ]:
from collections import Counter

from groundline.graph_view import gen_subgraph, subgraph_elements, enclosing_module_name

sub = gen_subgraph(ir, ent)
nodes, edges = subgraph_elements(ir, sub, ent)

print(f"{FOCUS_ID}: {len(nodes)} node elements, {len(edges)} edge elements")
print("edge confidence mix:",
      dict(Counter(e["data"].get("confidence", "n/a") for e in edges)))
print("enclosing module:", enclosing_module_name(ir, ent.id))
print()
print("first node element:", nodes[0])
print("first edge element:", edges[0])

Element dicts carry the facts as `data` attributes (`id`, `label`, `type`,
`defined`, `parent` for module grouping; `relation`, `direction`, `confidence`
on edges) — which is exactly what the widget's stylesheet selects on. Rendering
elsewhere (a report, another widget toolkit) starts from these dicts.

As a taste of using them analytically: the busiest neighbourhoods in the corpus,
by element count.

In [ ]:
from groundline.ir import SUBROUTINE, FUNCTION

degree = Counter()
for caller, callee in ir.calls:
    degree[caller] += 1
    degree[callee] += 1
busiest = [eid for eid, _ in degree.most_common(5)]
for eid in busiest:
    e = ir.get(eid)
    if e is None or e.kind not in (SUBROUTINE, FUNCTION):
        continue
    n, ed = subgraph_elements(ir, gen_subgraph(ir, e), e)
    mix = Counter(el["data"].get("confidence", "membership") for el in ed)
    print(f"{eid:55} {len(n):4} nodes, {len(ed):4} edges  {dict(mix)}")

## Where to go next

* **03_module_dependencies** — zoom out to module-level structure.
* **04_confidence_queries** — query the confidence strata: must-vs-may
  reachability, unresolved-target censuses, invariant sketches.